# 01 — El modelo base preentrenado

**Módulo:** `src/llm_rlhf/model.py` → `PretrainedLLM`

Antes de poder *alinear* un modelo de lenguaje, necesitamos uno. Este notebook es el más corto de la serie porque la idea es corta: un LLM preentrenado es simplemente una función de texto a texto. Todo lo que sigue — SFT, modelado de recompensa, PPO/DPO/GRPO — se trata de *moldear* esa función para que sus salidas sean más útiles.

## Setup (Colab o entorno nuevo)

Esta primera celda es **idempotente**: si `llm_rlhf` ya está instalado, no hace nada. En Colab, la primera vez clona el repositorio y lo instala en modo editable.

El repositorio apuntado es [emiliomunozai/LLM_RLHF](https://github.com/emiliomunozai/LLM_RLHF).

In [ ]:
# --- Colab / fresh-environment setup ---------------------------------
# No-op if llm_rlhf is already importable (e.g. local uv environment).
import os, sys, subprocess

REPO_URL = "https://github.com/emiliomunozai/LLM_RLHF.git"
REPO_DIR = "LLM_RLHF"

try:
    import llm_rlhf  # noqa: F401
except ImportError:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("Installed llm_rlhf. If torch/transformers were upgraded, restart the runtime now.")

## ¿Qué hace en realidad un LLM preentrenado?

Fue entrenado con *predicción del siguiente token*: dada una ventana de texto, predecir el siguiente token. Ese es todo el objetivo de entrenamiento. Como efecto secundario de hacer esto sobre Internet, el modelo absorbe gramática, hechos y patrones de razonamiento.

Lo que **nunca se le mostró** es la diferencia entre *responder a una pregunta* y *continuar un texto que casualmente parece una pregunta*. Por eso, cuando le das una instrucción, puede:

- continuar con una lista de *más* preguntas,
- dar una respuesta tipo Wikipedia que ignora a su audiencia,
- o irse por las ramas hacia texto irrelevante.

El primer paso de alineamiento (SFT, notebook 02) le enseñará el *formato* de pregunta-respuesta. Los pasos posteriores (PPO/DPO/GRPO) le enseñarán cuáles respuestas son *mejores*.

> **Intuición clave:** un LLM no "sabe responder"; sabe *qué texto suele venir después de otro texto*. La alineación es enseñarle qué cuenta como "buena continuación" cuando lo anterior es una instrucción.

## ¿Por qué OPT-350M?

Por defecto usamos `facebook/opt-350m` porque es el modelo más pequeño que:

- aún produce inglés coherente (para que podamos *ver* diferencias cualitativas tras la alineación), y
- cabe en una GPU de portátil en fp16 (~700 MB) para que puedas correr el pipeline completo.

Cada módulo del repositorio es agnóstico al modelo — el wrapper expone la misma interfaz para cualquier LM causal de Hugging Face.

In [1]:
from llm_rlhf import PretrainedLLM

llm = PretrainedLLM()  # facebook/opt-350m by default

Loading facebook/opt-350m on cuda...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

Loaded 331.2M parameters.


## El modelo en tres líneas

`PretrainedLLM` es intencionalmente diminuto — solo hace tres cosas que necesitarás más adelante:

1. `generate(prompt)` — muestrea una continuación
2. `save_checkpoint(path)` — persiste pesos + tokenizer
3. `load_adapter(path)` — engancha un adaptador LoRA entrenado por una etapa posterior

Nada más. Cualquier cosa más interesante vive en un módulo aguas abajo.

## Nuestro caso de seguimiento

A lo largo de **todas las notebooks** evaluaremos el modelo sobre el mismo prompt canónico:

> `"Explain quantum computing to a 10-year-old."`

Exportado como `CANONICAL_PROMPT` desde `llm_rlhf.eval`. La idea es que después de cada etapa (SFT → RM → PPO/DPO/GRPO) vuelvas a generar sobre este mismo prompt para *ver* el cambio. Aquí, partiendo del modelo base sin alinear, esperamos un comportamiento muy distinto al de un asistente real — el modelo *no sabe* que se espera una respuesta.

In [ ]:
# What does an unaligned model do with an instruction?
from llm_rlhf import CANONICAL_PROMPT

print(llm.generate(CANONICAL_PROMPT))

## Qué observar en la salida

Si corriste la celda anterior sobre un OPT-350M recién descargado, probablemente obtuviste (a) una definición tipo libro de texto de la computación cuántica, o (b) texto que se desvía hacia un *non sequitur*. Ambos son *fallos de seguimiento de instrucciones*, no fallos lingüísticos.

**Esta es la brecha que RLHF cierra.** Cada notebook posterior cierra una parte de esa brecha.

## Visualizando la predicción del siguiente token

La afirmación "un LLM es una función que asigna una distribución sobre el siguiente token" suena abstracta. Hagámosla concreta: tomemos un prompt y miremos los 10 tokens más probables que el modelo *cree* que vienen a continuación, junto con sus probabilidades.

Esto te muestra el "estado mental" del modelo en un instante: no genera un token, genera una *distribución*, y luego muestrea de ella.

In [ ]:
import matplotlib.pyplot as plt
from llm_rlhf import CANONICAL_PROMPT

def plot_next_token_distribution(
    llm,
    prompt: str,
    top_k: int = 20,
    temperature: float = 0.0,
    top_p: float = 1,
    ax=None,
):
    """Show the top-k candidate next tokens after temperature + top-p filtering.

    `temperature` and `top_p` use the same math as `llm.generate()`:
    - temperature < 1 sharpens the distribution, > 1 flattens it
    - top_p < 1 zeros out the tail (those tokens cannot be sampled)
    """
    tokens, probs = llm.next_token_distribution(
        prompt, temperature=temperature, top_p=top_p, top_k=top_k
    )
    # Reverse so the most-likely token sits at the top of the bar chart
    tokens = tokens[::-1]
    probs = probs.flip(0).numpy()

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(range(top_k), probs)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([repr(t) for t in tokens])
    ax.set_xlabel("probability")
    ax.set_xlim(0, 1)
    ax.set_title("T={:.2f}, top_p={:.2f}".format(temperature, top_p))
    if own_fig:
        fig.suptitle("Top-{} next tokens after: {!r}".format(top_k, prompt))
        plt.tight_layout()
        plt.show()

plot_next_token_distribution(llm, CANONICAL_PROMPT)

### Qué notar en el gráfico

Mira la *forma* de la distribución, no solo el token ganador:

- **Distribución plana** → el modelo está "indeciso"; muchos tokens son aproximadamente igual de plausibles. Esto es típico tras un prompt abierto.
- **Distribución concentrada** → el modelo está "seguro" de un único token. Esto ocurre dentro de frases formuladas ("Quantum computing **is**...").
- **Tokens raros** → fíjate cómo el tokenizer parte palabras y espacios; el primer token suele tener un espacio inicial (` quantum` vs `quantum`).

La *temperatura* en `generate()` reescala estas probabilidades antes de muestrear: temperatura alta aplana la distribución (más diversidad), temperatura baja la agudiza (más determinismo).

## El efecto de `temperature` y `top_p`

Hasta ahora vimos la distribución *cruda* (T=1, top_p=1). Pero en la práctica nunca muestreamos de esa distribución directamente — `generate()` siempre aplica dos transformaciones antes:

- **Temperature (T)**: divide los logits por `T` antes del softmax.
  - `T < 1` → distribución más *picuda*: el modelo "se compromete" con el token más probable.
  - `T > 1` → distribución más *plana*: tokens raros tienen más oportunidad.
  - `T → 0` → equivale a `argmax` (greedy decoding, determinista).

- **Top-p (nucleus sampling)**: tras el softmax, ordena los tokens por probabilidad descendente y conserva *solo* el conjunto más pequeño cuya masa acumulada alcanza `top_p`; al resto les asigna probabilidad cero. Esto **corta la cola** y evita muestrear tokens improbables sin tener que fijar un `top_k` rígido.

A continuación graficamos la misma posición del prompt bajo varias combinaciones para que veas cómo cambia la *forma* de la distribución. Fíjate en:

1. cómo cambia la **altura de la barra superior** al variar `T` (más alta = más confianza),
2. cómo `top_p` **vacía las barras inferiores** (probabilidad exactamente 0),
3. y cómo `T` y `top_p` interactúan: bajar `T` ya sesga hacia el top — añadir `top_p` bajo lo refuerza.

In [ ]:
from llm_rlhf import CANONICAL_PROMPT

# Each row sweeps temperature at a fixed top_p so you can read off both axes.
settings = [
    [(0.3, 1.0), (1.0, 1.0), (1.5, 1.0)],   # top_p=1.0 (no truncation)
    [(0.3, 0.5), (1.0, 0.5), (1.5, 0.5)],   # top_p=0.5 (aggressive nucleus)
]

fig, axes = plt.subplots(
    len(settings), len(settings[0]), figsize=(15, 8), sharex=True
)
for row, row_settings in enumerate(settings):
    for col, (T, p) in enumerate(row_settings):
        plot_next_token_distribution(
            llm, CANONICAL_PROMPT, top_k=10, temperature=T, top_p=p, ax=axes[row, col]
        )
fig.suptitle("Next-token distribution under different (T, top_p) — prompt: {!r}".format(CANONICAL_PROMPT))
plt.tight_layout()
plt.show()

## Ejercicio: ¿qué tan "instructable" es el modelo base?

Compara cómo se comporta el modelo frente a tres tipos distintos de entrada. Antes de correr la siguiente celda, **predice** qué crees que pasará en cada caso. Luego ejecuta y compara:

1. **Una instrucción directa** — ¿la sigue, o la trata como texto a continuar?
2. **Un comienzo de historia** — aquí el modelo *debería* brillar (es exactamente para lo que fue entrenado).
3. **Una pregunta factual corta** — ¿da una respuesta, o sigue generando más preguntas?

Después, prueba a cambiar la temperatura (`GenerationConfig(temperature=...)`) y observa cómo varía la diversidad.

In [ ]:
from llm_rlhf import EVAL_PROMPTS
from llm_rlhf.model import GenerationConfig

# EVAL_PROMPTS = [canonical instruction, factual question, creative instruction]
# We'll re-evaluate exactly these in 02/04/05/06 so the progression is visible.
labels = ["instruction (canonical)", "factual question", "creative instruction"]

cfg = GenerationConfig(max_new_tokens=60, temperature=0.7)
for label, p in zip(labels, EVAL_PROMPTS):
    print("--- {} ---".format(label))
    print("PROMPT:    ", p)
    print("COMPLETION: ", llm.generate(p, cfg))
    print()

## Cierre — guarda esta línea base

Lo que acabas de ver es **el punto cero** de la alineación. La salida del modelo sobre `CANONICAL_PROMPT` arriba — incoherente, divagatoria, sin la forma de una respuesta — es la *línea base* contra la que compararemos cada etapa siguiente.

Mentalmente, marca tres cosas para volver a evaluarlas tras SFT, PPO/DPO/GRPO:

1. **¿Responde la pregunta?** (instrucción canónica)
2. **¿Da un hecho razonable?** (pregunta factual)
3. **¿Sigue una restricción creativa específica?** (haiku)

## Siguiente: `02_sft.ipynb`

Haremos fine-tuning de este modelo sobre Dolly-15k para que aprenda la *forma* de un intercambio instrucción → respuesta. Volveremos a generar sobre `CANONICAL_PROMPT` al final y la diferencia debería ser inmediata: aunque el contenido siga siendo de OPT-350M (no es ChatGPT), la *estructura* será la de una respuesta.